In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import *

# ===============================
# 1. LOAD RAW DATA (Bronze Layer)
# ===============================
df_customers = spark.read.parquet('abfss://insurance@onelake.dfs.fabric.microsoft.com/insurance_lakehouse.Lakehouse/Files/Bronze/customer/customers.parquet')
df_policies  = spark.read.parquet('abfss://insurance@onelake.dfs.fabric.microsoft.com/insurance_lakehouse.Lakehouse/Files/Bronze/policy/policies.parquet')
df_claims    = spark.read.parquet('abfss://insurance@onelake.dfs.fabric.microsoft.com/insurance_lakehouse.Lakehouse/Files/Bronze/claim/claims.parquet')


StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 3, Finished, Available, Finished)

In [2]:
display(df_customers.limit(10))

StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 4, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, dc87f2b5-417a-4e0e-84c6-6902baaa78d0)

In [3]:
display(df_policies.limit(10))

StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, fac22908-ff16-41b8-a7fa-e26c77813eb5)

In [4]:
display(df_claims.limit(10))

StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 6, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 025d02bd-3599-42d4-aa5f-e22d5adc37d8)

# Cleaning the customer data

In [5]:
display(df_customers.limit(5))

StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 7, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 6468090b-4a7b-4e07-8527-12afe68bea6f)

In [6]:
df_customers_clean = (
    df_customers
    .dropna(subset=["cust_id"])
    .withColumn("cust_id", col("cust_id").cast("int"))
    .withColumn("CustomerName", initcap(trim(col("Full Name"))))
    .withColumn("Date_of_Birth", to_date(col("Date_of_Birth")))
    .withColumn("Contact", when(col("Contact").isNull(), "Not Provided").otherwise(col("Contact")))
    .drop("Full Name")
    .dropDuplicates()
)
display(df_customers_clean)

StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 8, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 21414e37-1784-4387-826d-4d1d51e6a940)

# Cleaning the policy data

In [7]:
display(df_policies.limit(10))

StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, c4780949-79f4-45dc-956b-92511db155d9)

In [13]:
df_policies_clean = (
    df_policies
    .withColumn("policy_Type", initcap(trim(col("policy_Type"))))
    .withColumn("status", initcap(trim(col("status"))))
    .withColumn("Start_Date", to_date(col("Start_Date"), "dd/MM/yyyy"))
    .withColumn("Coverage_Amount", when(col("Coverage_Amount") == "not available", None)
                .otherwise(col("Coverage_Amount").cast("double")))
    .withColumn("cust_id", col("cust_id").cast("int"))
    .dropna(subset=["cust_id", "policy_id"])
)
display(df_policies_clean)

StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 15, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 42d3309d-1e12-48fb-ab55-ba809522d3cc)

# Cleaning the Claim data

In [15]:
display(df_claims.limit(5))

StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 17, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 3c3e6d0d-17c9-4c3e-b1be-51edbb663997)

In [16]:
df_claims_clean = (
    df_claims
    .withColumn("claim_date", to_date(col("claim_date")))
    .withColumn("claim_amount", col("claim_amount").cast("double"))
    .withColumn("status", initcap(trim(col("status"))))
    .dropna(subset=["policy_id"])
)
display(df_claims_clean)

StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 18, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, af52c803-ce9b-4452-b5f8-9d765330b239)

# create delta silver table for this

In [27]:
df_customers_clean.write.mode("overwrite").format("delta").saveAsTable("silver_customers")
df_policies_clean.write.mode("overwrite").format("delta").saveAsTable("silver_policies")
df_claims_clean.write.mode("overwrite").format("delta").saveAsTable("silver_claims")


StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 29, Finished, Available, Finished)

In [30]:
%%sql 
select * from insurance_lakehouse.silver_customers

StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 31, Finished, Available, Finished)

<Spark SQL result set with 50 rows and 5 fields>

# gold tables 

In [31]:
df_joined = (
    df_claims_clean.alias("c")
    .join(df_policies_clean.alias("p"), col("c.policy_id") == col("p.policy_id"), "inner")
    .join(df_customers_clean.alias("cu"), col("p.cust_id") == col("cu.cust_id"), "inner")
)
display(df_joined.limit(5))


StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 32, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 9b1bafb6-20e4-4a7d-a973-ab2afac3af68)

In [33]:
df_aggregated = (
    df_joined.groupBy("cu.cust_id", "cu.CustomerName", "cu.Gender")
    .agg(
        count("c.claim_id").alias("TotalClaims"),
        count(when(lower(col("c.status")) == "approved", True)).alias("ApprovedClaims"),
        count(when(lower(col("c.status")) == "rejected", True)).alias("RejectedClaims"),
        round(sum("c.claim_amount"), 2).alias("TotalClaimAmount"),
        round(avg("c.claim_amount"), 2).alias("AvgClaimAmount"),
        collect_set(lower(col("p.policy_type"))).alias("PolicyTypesArray"),
        round(sum("p.coverage_amount"), 2).alias("TotalCoverageAmount"),
        min("c.claim_date").alias("FirstClaimDate"),
        max("c.claim_date").alias("LastClaimDate")
    )
    .withColumn("PolicyTypes", concat_ws(", ", col("PolicyTypesArray")))
    .withColumn(
        "ClaimToCoverageRatio",
        round((col("TotalClaimAmount") / col("TotalCoverageAmount")) * 100, 2)
    )
    .drop("PolicyTypesArray")
)
display(df_aggregated)

# SAVE the cleaned+enriched silver output
df_aggregated.write.mode("overwrite").format("delta").save("abfss://insurance@onelake.dfs.fabric.microsoft.com/insurance_lakehouse.Lakehouse/Files/gold")


StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 34, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, ed7c03b5-eb69-482f-b8be-8b3a47c9529e)

In [34]:
df_aggregated.write.mode("overwrite").format("delta").saveAsTable("insurance_aggrgate")

StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 35, Finished, Available, Finished)

In [35]:
%%sql 
select * from insurance_lakehouse.insurance_aggrgate

StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 36, Finished, Available, Finished)

<Spark SQL result set with 24 rows and 13 fields>

# Top 10 High-Value Customers by Claim Amount

In [36]:
display(spark.sql("""
SELECT CustomerName, TotalClaimAmount
FROM  insurance_lakehouse.insurance_aggrgate
ORDER BY TotalClaimAmount DESC
LIMIT 10
"""))

StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 37, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 187601ac-eb27-4fd5-8097-6ac712894218)

# Claim Approval Rate by Gender

In [37]:
display(spark.sql("""
SELECT Gender,
       ApprovedClaims,
       RejectedClaims,
       ROUND(ApprovedClaims * 100.0 / TotalClaims, 2) AS ApprovalRate
FROM insurance_lakehouse.insurance_aggrgate
"""))


StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 38, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 0bf784dc-f998-4b73-8a95-6db2e7dce48c)

 # 3. Claim to Coverage Ratio Distribution

In [38]:
display(spark.sql("""
SELECT CustomerName, ClaimToCoverageRatio
FROM insurance_lakehouse.insurance_aggrgate
WHERE ClaimToCoverageRatio IS NOT NULL
ORDER BY ClaimToCoverageRatio DESC
"""))

StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 39, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 284f1a93-a8b4-4218-8284-f941522484c8)

# Claims by Policy Types (Explode for Analysis)

In [39]:
from pyspark.sql.functions import explode, split

df_policy_split = (
    df_aggregated
    .withColumn("PolicyType", explode(split(col("PolicyTypes"), ",\\s*")))
    .groupBy("PolicyType")
    .agg(count("*").alias("CustomerCount"))
)

display(df_policy_split)

StatementMeta(, b9678606-628e-401b-bc7d-46bc12566279, 40, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 9117ab4f-2499-41aa-8940-1db4425e61ad)